In [ ]:
import pandas as pd
import kagglehub
from pathlib import Path

# Cargar Global Superstore
try:
    print("Cargando dataset desde Kaggle...")
    dataset_path = Path(
        kagglehub.dataset_download("rohitgrewal/global-superstore-data")
    )

    xlsx_files = list(dataset_path.rglob("*.xlsx"))
    xlsx_files += list(dataset_path.rglob("*.xls"))

    if not xlsx_files:
        raise FileNotFoundError("No se encontraron archivos Excel en Kaggle.")
    pages = pd.read_excel(xlsx_files[0], sheet_name=None)
    pages_names = {
        name.strip().lower(): name
        for name in pages.keys()
    }
    required = ["orders", "people", "returns"]

    if not all(name in pages_names for name in required):
        raise ValueError(
            f"No se encontraron todas las hojas requeridas. "
            f"Hojas disponibles: {list(pages.keys())}"
        )
    df_orders = pages[pages_names["orders"]]
    df_people = pages[pages_names["people"]]
    df_returns = pages[pages_names["returns"]]

    print("Global Superstore cargado correctamente desde Kaggle.")

except Exception as error:
    print(f"Error al cargar desde Kaggle: {error}")
    print("Intentando cargar el archivo Excel local...")
    local_file = Path("Global Superstore Data.xlsx")
    if not local_file.exists():
        raise FileNotFoundError(
            f"No se encontró el archivo local: {local_file.resolve()}"
        )
    pages = pd.read_excel(local_file, sheet_name=None)
    pages_names = {
        name.strip().lower(): name
        for name in pages.keys()
    }

    required = ["orders", "people", "returns"]
    if not all(name in pages_names for name in required):
        raise ValueError(
            f"No se encontraron todas las hojas requeridas. "
            f"Hojas disponibles: {list(pages.keys())}"
        )
    df_orders = pages[pages_names["orders"]]
    df_people = pages[pages_names["people"]]
    df_returns = pages[pages_names["returns"]]
    print("Global Superstore cargado correctamente desde el archivo local.")


# Cargar Logistics Performance Index (LPI)
try:
    print("\nCargando datos del LPI desde World Bank Data360...")
    # Configure the direct CSV download URL when available
    lpi_url = "https://data360files.worldbank.org/data360-data/data/WB_LPI/LPI_OVR.csv"
    if lpi_url is None:
        raise ValueError("No se ha configurado la URL directa del CSV del LPI.")
    df_lpi = pd.read_csv(lpi_url)
    if df_lpi.empty:
        raise ValueError("La fuente en línea no contiene registros.")
    print("LPI cargado correctamente desde World Bank Data360.")

except Exception as error:
    print(f"Error al cargar el LPI en línea: {error}")
    print("Intentando cargar el archivo LPI_OVR.csv local...")
    local_lpi = Path("LPI_OVR.csv")
    if not local_lpi.exists():
        raise FileNotFoundError(
            f"No se encontró el archivo local del LPI: {local_lpi.resolve()}"
        )
    df_lpi = pd.read_csv(local_lpi)
    print("LPI cargado correctamente desde el archivo local.")


# Mostrar resumen de los datos cargados
print("\nResumen de los datos cargados:")
print(f"Órdenes: {df_orders.shape}")
print(f"Personal: {df_people.shape}")
print(f"Devoluciones: {df_returns.shape}")
print(f"LPI: {df_lpi.shape}")

Cargando dataset desde Kaggle...


100%|██████████| 6.93M/6.93M [00:00<00:00, 31.4MB/s]

Extracting files...


Global Superstore cargado correctamente desde Kaggle.

Cargando datos del LPI desde World Bank Data360...
LPI cargado correctamente desde World Bank Data360.

Resumen de los datos cargados:
Órdenes: (51290, 24)
Personal: (24, 2)
Devoluciones: (2033, 3)
LPI: (19356, 41)


# 1. Limpieza y Transformación

In [ ]:
import numpy as np
# Limpieza y Transformación

# 1. Estandarizar columnas de texto
text_col_orders = [
    "Order ID", "Ship Mode", "Customer ID",
    "Customer Name", "Segment", "City",
    "State", "Country", "Region",
    "Market", "Product ID", "Product Name",
    "Sub-Category", "Category", "Order Priority"
]

for col in text_col_orders:
    df_orders[col] = df_orders[col].str.lower().str.strip()

country_map = {
    'turkey': 'turkiye',
    'iran': 'iran, islamic rep.',
    'egypt': 'egypt, arab rep.',
    'russia': 'russian federation',
    'democratic republic of the congo': 'congo, dem. rep.',
    'vietnam': 'viet nam',
    'venezuela': 'venezuela, rb',
    'south korea': 'korea, rep.',
    'myanmar (burma)': 'myanmar',
    'côte-d\'ivoire': "cote d'ivoire",
    'czech republic': 'czechia',
    'kyrgyzstan': 'kyrgyz republic',
    'yemen': 'yemen, rep.',
    'syria': 'syrian arab republic',
    'martinique': 'france',
    'hong kong': 'hong kong sar, china',
    'taiwan': 'taiwan, china',
    'uganda': 'uganda',
    'albania': 'albania',
    'sierra leone': 'sierra leone',
    'trinidad and tobago': 'trinidad and tobago',
    'slovakia': 'slovak republic',
    'guadeloupe': 'france',
    'republic of the congo': 'congo, rep.',
    'macedonia': 'north macedonia',
    'french guiana': 'france',
    'laos': 'lao pdr',
    'the gambia': 'gambia, the',
    'swaziland': 'eswatini'
}


df_orders["Country"] = df_orders["Country"].replace(country_map)

df_orders["Region"] = df_orders["Region"].replace({"canada": "eastern canada"}) # Estandarización de Región

# 2. Convertir fechas a datetime
df_orders["Order Date"] = pd.to_datetime(df_orders["Order Date"])
df_orders["Ship Date"] = pd.to_datetime(df_orders["Ship Date"])

# 3. Extraer componentes de fecha
df_orders["Month"] = df_orders["Order Date"].dt.month
df_orders["Year"] = df_orders["Order Date"].dt.year

# 4. Crear período Año-Mes
df_orders["Year-Month"] = df_orders["Order Date"].dt.to_period("M")

# 5. Calcular margen de rentabilidad
df_orders["Margen"] = np.where(
    df_orders["Sales"] != 0,
    df_orders["Profit"] / df_orders["Sales"],
    np.nan
)

# 6. Eliminar columna innusable en el estudio
df_orders = df_orders.drop(columns=["Postal Code"])

# 2. Cruces Relacionales


In [ ]:
# CRUCES RELACIONALES CORREGIDOS

# 1. Preparar ORDERS
df_merged = df_orders.copy()

for columna in ["Order ID", "Country", "Region"]:
    df_merged[columna] = (
        df_merged[columna]
        .astype("string")
        .str.lower()
        .str.strip()
    )

# 2. MERGE ORDERS + RETURNS (Eliminando duplicados de Order ID en Returns)
df_returns_preparado = df_returns[["Order ID", "Returned"]].copy()

for columna in ["Order ID", "Returned"]:
    df_returns_preparado[columna] = (
        df_returns_preparado[columna]
        .astype("string")
        .str.lower()
        .str.strip()
    )

df_returns_preparado = df_returns_preparado.drop_duplicates(subset=["Order ID"])

df_merged = pd.merge(
    df_merged,
    df_returns_preparado,
    on="Order ID",
    how="left"
)

df_merged["Returned"] = (
    df_merged["Returned"]
    .fillna("no")
    .str.lower()
    .str.strip()
)

print("\nEstado de devoluciones (sin inflar filas):")
print(df_merged["Returned"].value_counts(dropna=False))

# 3. MERGE ORDERS + PEOPLE
df_people_preparado = df_people.copy()

for columna in ["Region", "Person"]:
    df_people_preparado[columna] = (
        df_people_preparado[columna]
        .astype("string")
        .str.lower()
        .str.strip()
    )

df_merged = pd.merge(
    df_merged,
    df_people_preparado,
    on="Region",
    how="left"
)

# 4. PREPARAR LPI
df_lpi_preparado = df_lpi.copy()

columnas_texto_lpi = [
    "INDICATOR", "COMP_BREAKDOWN_1", "COMP_BREAKDOWN_2",
    "COMP_BREAKDOWN_3", "UNIT_MEASURE", "REF_AREA_LABEL"
]

for columna in columnas_texto_lpi:
    df_lpi_preparado[columna] = (
        df_lpi_preparado[columna]
        .astype("string")
        .str.lower()
        .str.strip()
    )

# Filtrar únicamente el LPI Overall Score
df_lpi_preparado = df_lpi_preparado[
    (df_lpi_preparado["INDICATOR"] == "lpi_ovr") &
    (df_lpi_preparado["COMP_BREAKDOWN_1"] == "_t") &
    (df_lpi_preparado["COMP_BREAKDOWN_2"] == "wb_lpi_sc") &
    (df_lpi_preparado["COMP_BREAKDOWN_3"] == "_z") &
    (df_lpi_preparado["UNIT_MEASURE"] == "score")
].copy()

# 5. Renombrar y convertir tipos
df_lpi_preparado = df_lpi_preparado.rename(
    columns={
        "REF_AREA_LABEL": "Country",
        "TIME_PERIOD": "Year",
        "OBS_VALUE": "LPI_Observado"
    }
)

df_lpi_preparado["Year"] = pd.to_numeric(df_lpi_preparado["Year"], errors="coerce")
df_lpi_preparado["LPI_Observado"] = pd.to_numeric(df_lpi_preparado["LPI_Observado"], errors="coerce")
df_lpi_preparado["Country"] = df_lpi_preparado["Country"].astype("string").str.lower().str.strip()

# 6. CORTAR SERIE EN 2018 (Regla #4: No usar 2023 para interpolar hacia atrás)
# Usamos el rango completo de benchmarks pares (2010 a 2018) para que los años impares
# (como 2011, 2013, 2015, 2017) tengan siempre su año par anterior y posterior real.
min_year_lpi = 2010
max_year_lpi = 2018
rango_anual_lpi = list(range(min_year_lpi, max_year_lpi + 1))

df_lpi_preparado = df_lpi_preparado[
    df_lpi_preparado["Year"].between(min_year_lpi, max_year_lpi)
].copy()

# 7. Crear estructura completa Country + Year (2010 a 2018)
paises_lpi = df_lpi_preparado["Country"].dropna().unique()

df_lpi_completo = (
    pd.MultiIndex
    .from_product([paises_lpi, rango_anual_lpi], names=["Country", "Year"])
    .to_frame(index=False)
)

# 8. Incorporar valores observados reales del LPI (años pares)
df_lpi_completo = pd.merge(
    df_lpi_completo,
    df_lpi_preparado[["Country", "Year", "LPI_Observado"]],
    on=["Country", "Year"],
    how="left"
).sort_values(["Country", "Year"])

# 9. Interpolar linealmente (SOLO hacia adentro entre puntos observados primero)
df_lpi_completo["LPI_Interpolado_Lineal"] = (
    df_lpi_completo
    .groupby("Country")["LPI_Observado"]
    .transform(lambda s: s.interpolate(method="linear", limit_area="inside"))
)

# Para países que no tuvieron encuesta en un extremo (ej. 2010 o 2018), propagar extremos como fallback
df_lpi_completo["LPI"] = (
    df_lpi_completo
    .groupby("Country")["LPI_Interpolado_Lineal"]
    .transform(lambda s: s.bfill().ffill())
)

# 10. Documentar el método de cada registro (Reglas #1 y #5: Trazabilidad)
df_lpi_completo["LPI_Metodo"] = np.select(
    [
        df_lpi_completo["LPI_Observado"].notna(),
        df_lpi_completo["LPI_Interpolado_Lineal"].notna(),
        df_lpi_completo["LPI"].notna()
    ],
    [
        "observado (bienal)",
        "interpolado (lineal)",
        "extrapolado (extremo)"
    ],
    default="sin datos"
)

# 11. Cruce FINAL Orders + LPI (Conservando LPI_Observado, LPI y LPI_Metodo)
df_merged = pd.merge(
    df_merged,
    df_lpi_completo[["Country", "Year", "LPI_Observado", "LPI", "LPI_Metodo"]],
    on=["Country", "Year"],
    how="left"
)

df_merged["LPI_Metodo"] = df_merged["LPI_Metodo"].fillna("sin datos")

print(f"\nFilas totales (deben ser 51,290 exactas): {len(df_merged)}")
print("\nDistribución de la fuente del LPI en las órdenes:")
print(df_merged["LPI_Metodo"].value_counts())


Estado de devoluciones (sin inflar filas):
Returned
no     45538
yes     5752
Name: count, dtype: Int64

Filas totales (deben ser 51,290 exactas): 51290

Distribución de la fuente del LPI en las órdenes:
LPI_Metodo
interpolado (lineal)     28352
observado (bienal)       22588
extrapolado (extremo)      315
sin datos                   35
Name: count, dtype: int64


# 3. Checklist de Auditoria de Datos previo al Analisis Exploratorio
Estándar Corporativo de Auditoría de Datos (ETL - Fase 1) Antes de hacer cualquier cruce o
limpieza profunda, cada tabla individual debe pasar por esta auditoría obligatoria en Python:
1. [ ] Inspección de Tamaño: ¿Cuántas filas y columnas reales tenemos? (df.shape)
2. [ ] Mapeo de Tipos de Datos: ¿Están los números como int/float y las fechas como
datetime, o Python leyó todo como texto (object)? (df.info())
3. [ ] Detección de Nulos: ¿Cuántos valores faltantes hay por columna?
(df.isnull().sum())
4. [ ] Detección de Duplicados: ¿Tenemos transacciones clonadas por error del sistema?
(df.duplicated().sum())
5. Análisis de Lógica de Negocio: ¿Existen valores físicamente imposibles? (Ej. precios
negativos, edades de 150 años). (df.describe()

In [ ]:
def auditar(df, nombre):
  print(f"""
  Inspección de Tamaño:
  {df.shape}
  Mapeo de tipos de datos:
  {df.info()}
  Detección de nulos:
  {df.isnull().sum()}
  Detección de Duplicados:
  {df.duplicated().sum()}
  Describe del dataset:
  {df.describe()}
  """)

auditar(df_merged, "df_merged")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 32 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Row ID          51290 non-null  int64         
 1   Order ID        51290 non-null  string        
 2   Order Date      51290 non-null  datetime64[ns]
 3   Ship Date       51290 non-null  datetime64[ns]
 4   Ship Mode       51290 non-null  object        
 5   Customer ID     51290 non-null  object        
 6   Customer Name   51290 non-null  object        
 7   Segment         51290 non-null  object        
 8   City            51290 non-null  object        
 9   State           51290 non-null  object        
 10  Country         51290 non-null  string        
 11  Region          51290 non-null  string        
 12  Market          51290 non-null  object        
 13  Product ID      51290 non-null  object        
 14  Product Name    51290 non-null  object        
 15  Su

# 4. El EDA Estadístico es obligatorio en Python (La base científica)
Antes de abrir Power BI, deben usar Python para la matemática pesada: ejecutar .describe()
para ver distribuciones, calcular matrices de correlación numéricas para ver qué variables
interactúan, y usar diagramas de caja (Boxplots) de Seaborn para detectar valores atípicos
(outliers).La justificación de negocio:
Si detectan un outlier masivo (ej. una venta de un millón de dólares por error tipográfico), la
corrección y limpieza de ese registro debe quedar programada en el script de Python para que
sea reproducible.
Si lo intentan parchar en Power BI, el error seguirá existiendo en su base de datos original.

In [1]:
import seaborn as sns
import matplotlib.pyplot as plt

### Analisis EDA ###

# 1. Tratamiento de Nulos de Negocio

# Si no se registró un descuento, se considera que no hubo descuento.
df_merged["Discount"] = df_merged["Discount"].fillna(0)

# Si no se registró una devolución, se considera que el pedido no fue devuelto.
df_merged["Returned"] = df_merged["Returned"].fillna("no")

# Si no existe información en estas columnas de texto,
# se registra como "sin registrar".
df_merged["Customer Name"] = df_merged["Customer Name"].fillna("sin registrar")
df_merged["State"] = df_merged["State"].fillna("sin registrar")
df_merged["Sub-Category"] = df_merged["Sub-Category"].fillna("sin registrar")


# 2. Eliminación de registros críticos

# Estas columnas son necesarias para identificar y analizar correctamente
# cada venta.
key_cols = [
    "Row ID", "Order ID", "Order Date",
    "Ship Date", "Ship Mode", "Customer ID",
    "Segment", "City", "Country",
    "Market", "Region", "Product ID",
    "Category", "Product Name", "Sales",
    "Quantity", "Profit", "Shipping Cost",
    "Order Priority"
]

df_merged = df_merged.dropna(subset=key_cols)

# 3. Eliminacion de duplicados
df_merged = df_merged.drop_duplicates()

# 4. Prueba final de limpieza

print("Limpieza completada.")
print("Filas finales:", len(df_merged))

# Verificar registros duplicados
duplicados = df_merged.duplicated().sum()
print("Registros duplicados:", duplicados)

print("Nulos restantes por columna:")
display(df_merged.isnull().sum())

print("Resumen estadístico:")
display(df_merged.describe())

print("Tipos de datos:")
print(df_merged.dtypes)


# 5. Graficas para analizar datos
print("\nMatriz de correlación de las variables numéricas")
correlacion = df_merged.corr(numeric_only=True)
display(correlacion)


def boxplot_resumen(df, col):
    datos = df[col].dropna()
    q1 = datos.quantile(0.25)
    mediana = datos.median()
    q3 = datos.quantile(0.75)
    iqr = q3 - q1
    li = q1 - 1.5 * iqr
    ls = q3 + 1.5 * iqr
    outliers = ((datos < li) | (datos > ls)).sum()

    plt.figure(figsize=(10, 4))
    sns.boxplot(x=datos)
    plt.title(f"Boxplot de {col}")
    plt.show()

    print(f"{col}: Q1={q1:.2f}, Mediana={mediana:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}, Outliers={outliers}")

boxplot_resumen(df_merged, "Sales")
boxplot_resumen(df_merged, "Profit")

# Comparar correlación usando LPI Interpolado (Todos los años) vs LPI Observado (Solo años pares)
cols_interes = ["Sales", "Profit", "Shipping Cost", "Margen"]

print("--- Correlación con LPI Anualizado (Interpolación Lineal) ---")
display(df_merged[cols_interes + ["LPI"]].corr()[["LPI"]])

print("\n--- Correlación con LPI Observado (Solo Años Pares - Sin Inventar Datos) ---")
display(df_merged[cols_interes + ["LPI_Observado"]].corr()[["LPI_Observado"]])


NameError: name 'df_merged' is not defined

#Checklist de Auditoria de Datos posterior al Analisis Exploratorio


In [ ]:
auditar(df_merged, "Post-EDA")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 32 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Row ID          51290 non-null  int64         
 1   Order ID        51290 non-null  string        
 2   Order Date      51290 non-null  datetime64[ns]
 3   Ship Date       51290 non-null  datetime64[ns]
 4   Ship Mode       51290 non-null  object        
 5   Customer ID     51290 non-null  object        
 6   Customer Name   51290 non-null  object        
 7   Segment         51290 non-null  object        
 8   City            51290 non-null  object        
 9   State           51290 non-null  object        
 10  Country         51290 non-null  string        
 11  Region          51290 non-null  string        
 12  Market          51290 non-null  object        
 13  Product ID      51290 non-null  object        
 14  Product Name    51290 non-null  object        
 15  Su

In [ ]:
# Exportar el dataset final a Excel
from google.colab import files

df_merged.to_excel("Global_Superstore_Limpio.xlsx", index=False)

# Descargar el archivo a tu computadora
files.download("Global_Superstore_Limpio.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>